In [3]:
import sys
sys.path.append('..')

import os
import random
import numpy as np
import cv2
import matplotlib.pyplot as plt

from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision import transforms
from torchvision.models.segmentation import deeplabv3_resnet50

In [4]:
print(os.getcwd())

/Users/jawaad/comp9517/groupAss/COMP9517_Project/DeepLabV3


In [5]:
# GPU Setup

if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
    
print(device)

mps


In [12]:
BASE_DIR = "Dataset"

TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR = os.path.join(BASE_DIR, "validation")
TEST_DIR = os.path.join(BASE_DIR, "test")

In [13]:
IMAGE_SIZE = 350
BATCH_SIZE = 4
EPOCHS = 15
LEARNING_RATE = 1e-4

NUM_CLASSES = 1

In [14]:
from DeepLabV3.WheatDataset import WheatDataset
from DeepLabV3.transformers import base_transform, aug_transform

train_dataset_base = WheatDataset(
    TRAIN_DIR,
    base_transform
)

train_dataset_aug = WheatDataset(
    TRAIN_DIR,
    aug_transform
)

val_dataset = WheatDataset(
    VAL_DIR,
    base_transform
)

test_dataset = WheatDataset(
    TEST_DIR,
    base_transform
)

In [15]:
train_loader_base = torch.utils.data.DataLoader(
    train_dataset_base,
    batch_size=BATCH_SIZE,
    shuffle=True
)

train_loader_aug = torch.utils.data.DataLoader(
    train_dataset_aug,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [16]:
def create_model():

    model = deeplabv3_resnet50(
        weights="DEFAULT"
    )
    
    model.classifier[4] = nn.Conv2d(
        256,
        NUM_CLASSES,
        kernel_size=1
    )

    return model.to(device)

In [18]:
from DeepLabV3.metrics import evaluate_model

def train_model(
    model,
    train_loader,
    val_loader,
    loss_fn,
    model_name
):

    optimizer = optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    best_iou = 0

    train_losses = []
    val_losses = []

    for epoch in range(EPOCHS):

        model.train()

        epoch_loss = 0

        for images, masks in tqdm(train_loader):

            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)['out']

            loss = loss_fn(
                outputs,
                masks
            )

            optimizer.zero_grad()

            loss.backward()

            optimizer.step()

            epoch_loss += loss.item()

        train_losses.append(epoch_loss)

        val_metrics = evaluate_model(
            model,
            val_loader,
            loss_fn,
            device
        )

        val_losses.append(val_metrics[0])

        precision, recall, f1, iou = val_metrics[1:]

        print(
            f"Epoch {epoch+1}/{EPOCHS}"
        )

        print(
            f"Precision: {precision:.4f}"
        )

        print(
            f"Recall: {recall:.4f}"
        )

        print(
            f"F1: {f1:.4f}"
        )

        print(
            f"IoU: {iou:.4f}"
        )

        if iou > best_iou:

            best_iou = iou

            torch.save(
                model.state_dict(),
                f"{model_name}.pth"
            )

    return train_losses, val_losses